In [4]:
import os

import pandas as pd
import numpy as np
import itertools
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.transforms import ScaledTranslation
from pathlib import Path

plt.rcParams['legend.handlelength'] = 1
plt.rcParams['legend.handleheight'] = 1.125
plt.rcParams["font.family"]         = "Avenir"

path_to_scenarios = '/Users/Guille/Desktop/india_power/scenarios'
path_to_images    = '/Users/Guille/Desktop/india_power/images'
path_to_tables    = '/Users/Guille/Desktop/india_power/tables'
path_to_csvs      = '/Users/Guille/Desktop/india_power/gridpath_india_viz/csvs'
path_to_inputs    = '/Users/Guille/Desktop/india_power/input_data'

# Production cost model - unserved energy

In [19]:
csv_ = pd.read_csv(path_to_csvs + '/pcm-scenario_labels.csv')

dfs_ = []
for scenario in csv_['scenario'].unique():
    df_ = pd.read_csv(path_to_scenarios + '/pcm/' + scenario + '/results/system_load_zone_timepoint.csv')

    label  = csv_.loc[csv_['scenario'] == scenario, 'label'].to_numpy()[0]
    period = df_['period'].unique()[0]
    
    OG = df_['overgeneration_mw'].sum()
    UE = df_['unserved_energy_mw'].sum()
    SL = df_['static_load_mw'].sum()
    
    dfs_.append([label, period, OG/1000., UE/1000., SL/1e6, 100.*OG/SL, 100.*UE/SL])

dfs_ = pd.DataFrame(dfs_, columns = ['Scenario', 
                                     'Period', 
                                     'Overgeneration (GWh)',
                                     'Unserved Energy (GWh)',
                                     'Load (TWh)',
                                     'Normalized Overgeneration (%)',
                                     'Normalized Unserved Energy (%)'])

dfs_['Period']                = dfs_['Period'].astype(int)
dfs_['Unserved Energy (GWh)'] = dfs_['Unserved Energy (GWh)'].astype(int)
dfs_['Overgeneration (GWh)']  = dfs_['Overgeneration (GWh)'].astype(int)
dfs_['Load (TWh)']            = dfs_['Load (TWh)'].astype(int)
dfs_['Normalized Overgeneration (%)']  = dfs_['Normalized Overgeneration (%)'].abs().round(3)
dfs_['Normalized Unserved Energy (%)'] = dfs_['Normalized Unserved Energy (%)'].round(3)

pivot_ = dfs_.pivot_table(index   = ['Scenario'],
                         columns = ['Period'],
                         values  = ['Unserved Energy (GWh)', 
                                    'Overgeneration (GWh)', 
                                    'Load (TWh)',
                                    'Normalized Overgeneration (%)',
                                    'Normalized Unserved Energy (%)']).reset_index(drop = False)
print(pivot_)

df_latex_ = pivot_.to_latex(index        = False,
                            float_format = lambda x: f'{x:,.3f}',           # all floats: 1,234.57
                            na_rep       = '--',
                            formatters   = {'Unserved Energy (GWh)': '{,.0f}'.format,
                                            'Overgeneration (GWh)': '{,.0f}'.format,
                                            'Load (TWh)': '{,.0f}'.format,
                                            'Normalized Overgeneration (%)': '{,.3f}'.format,
                                            'Normalized Unserved Energy (%)': '{,.3f}'.format})  # specific column (ints): 1,234
print(df_latex_)

Path(path_to_tables + f"/unserved_energy.tex").write_text(df_latex_, encoding="utf-8")

                                Scenario Load (TWh)              \
Period                                         2030  2040  2050   
0            VRE & ESS (high) Coal (low)       2571  4174  5603   
1             VRE & ESS (low) Coal (low)       2571  4174  5603   
2       VRE & ESS (mid) Coal (low) - REF       2571  4174  5603   

       Normalized Overgeneration (%)                \
Period                          2030   2040   2050   
0                                0.0  1.044  6.399   
1                                0.0  1.889  6.208   
2                                0.0  1.018  4.389   

       Normalized Unserved Energy (%)               Overgeneration (GWh)  \
Period                           2030   2040   2050                 2030   
0                               0.028  0.030  0.002                    0   
1                               0.030  0.006  0.004                    2   
2                               0.028  0.018  0.004                    0   

            

/var/folders/0c/ffx2kgyn7xq4krqrpsv67l3m0000gn/T/ipykernel_36207/3347926400.py:40: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  df_latex_ = pivot_.to_latex(index        = False,


1296

In [2]:
scenario = 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'

df_2_ = pd.read_csv(path_to_scenarios + f'/cost/{scenario}/inputs/transmission_lines.tab', sep='\t')
df_2_ = df_2_[df_2_['tx_capacity_type'] == 'tx_spec'].reset_index(drop = True)[['transmission_line',
                                                                                'tx_simple_loss_factor']]

df_1_ = pd.read_csv(path_to_tables + '/cost-tx_capacity.csv')

lz_ = pd.read_csv(path_to_inputs + '/load_zones.csv')
print(df_1_['scenario'].unique())


df_1_ = df_1_[df_1_['scenario'] == scenario].reset_index(drop = True)
df_1_ = df_1_[df_1_['period'] == 2020].reset_index(drop = True)

df_1_ = df_1_[['transmission_line', 
               'spec_mw']]

df_ = pd.merge(df_1_, df_2_, on='transmission_line')

df_['spec_mw'] = df_['spec_mw'].round().astype('Int64')
df_['tx_simple_loss_factor'] = 100.*df_['tx_simple_loss_factor']
df_['tx_simple_loss_factor'] = df_['tx_simple_loss_factor'].round(2)

df_.rename(columns = {'spec_mw': 'Capacity (MW)',
                      'transmission_line': 'Line',
                      'tx_simple_loss_factor': 'Loss Factor (%/km)'}, inplace = True)

for i in range(lz_.shape[0]):
    load_zone     = lz_.loc[i, 'load_zone']
    load_zone_abr = lz_.loc[i, 'load_zone_abr'].replace('JK', 'JKLA')
    df_['Line']   = df_['Line'].str.replace(load_zone, load_zone_abr)


df_latex_ = df_.to_latex(index=False,
                         float_format=lambda x: f'{x:,.2f}',           # all floats: 1,234.57
                         formatters={'Capacity (MW)': '{:,}'.format},  # specific column (ints): 1,234
                         na_rep='--')

# # Write to file and also print (so you can copy-paste)
Path(path_to_tables + f"/spec_tx.tex").write_text(df_latex_, encoding="utf-8")

['VREhigh_SThigh_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREhigh_SThigh_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VRElow_STlow_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVhigh_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']


/var/folders/0c/ffx2kgyn7xq4krqrpsv67l3m0000gn/T/ipykernel_27892/2838392597.py:35: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  df_latex_ = df_.to_latex(index=False,


NameError: name 'Path' is not defined

In [89]:
df_1_ = pd.read_csv(path_to_tables + '/cost-tx_transfers.csv')
print(df_1_['scenario'].unique())
period_  = [2030, 2040, 2050]
scenario = 'VRE & ESS (mid) Coal (low) - REF'

df_1_ = df_1_.loc[df_1_['scenario'] == scenario].reset_index(drop = True)
df_1_ = df_1_.loc[df_1_['period'].isin(period_)].reset_index(drop = True)

# df_2_ = pd.read_csv(path_to_tables + '/cost-tx_capacity.csv')
# print(df_2_)

# scenario = 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid'
# df_2_ = df_2_.loc[df_2_['scenario'] == scenario].reset_index(drop = True)
# df_2_ = df_2_.loc[df_2_['period'].isin(period_)].reset_index(drop = True)
# print(df_2_.shape)

df_ = df_1_.copy()
df_['net_transmission_flow_mw'] /= 1000

df_['net_transmission_flow_mw'] = df_['net_transmission_flow_mw'].round()
df_['max_transmission_flow_mw'] = df_['max_transmission_flow_mw'].round()
df_['min_transmission_flow_mw'] = df_['min_transmission_flow_mw'].round()

df_.rename(columns = {'transmission_line': 'Line',
                      'period': 'Period',
                      'net_transmission_flow_mw': 'Net transfers (TWh)',
                      'max_transmission_flow_mw': 'Max. transfer (MW)',
                      'min_transmission_flow_mw': 'Min. transfer (MW)'}, inplace = True)

lz_ = pd.read_csv(path_to_inputs + '/load_zones.csv')

for i in range(lz_.shape[0]):
    load_zone     = lz_.loc[i, 'load_zone']
    load_zone_abr = lz_.loc[i, 'load_zone_abr'].replace('JK', 'JKLA').replace('DH', 'DHDD')
    df_['Line']   = df_['Line'].str.replace(load_zone, load_zone_abr)

pivot_ = df_.pivot_table(index   = ['Line'],
                         columns = ['Period'],
                         values  = ['Net transfers (TWh)', 
                                    'Max. transfer (MW)', 
                                    'Min. transfer (MW)']).reset_index(drop = False)

df_latex_ = pivot_.to_latex(index        = False,
                            float_format = lambda x: f'{x:,.0f}',           # all floats: 1,234.57
                            na_rep       = '--',
                            formatters   = {'Net transfers (TWh)': '{:,}'.format,
                                            'Max. transfer (MW)': '{:,}'.format,
                                            'Min. transfer (MW)': '{:,}'.format})  # specific column (ints): 1,234
#print(df_latex_)

Path(path_to_tables + f"/transfers-ref.tex").write_text(df_latex_, encoding="utf-8")

['VRE & ESS (low) Coal (low)' 'VRE & ESS (low) Coal (high)'
 'VRE & ESS (mid) Coal (low) - REF' 'VRE & ESS (mid) Coal (high)'
 'VRE & ESS (high) Coal (low)' 'VRE & ESS (high) Coal (high)']


/var/folders/0c/ffx2kgyn7xq4krqrpsv67l3m0000gn/T/ipykernel_13771/849490788.py:43: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  df_latex_ = pivot_.to_latex(index        = False,


9792

In [73]:
import pandas as pd
from pathlib import Path

# --- Inputs ---
#CSV = 'cost-grouped_capacity' 
#CSV = 'additional-grouped_capacity' 
#CSV = 'demand-grouped_capacity'
#CSV = 'alternative-grouped_capacity'
#CSV = 'pier-grouped_capacity'
CSV = 'iced-grouped_capacity'

# --- Load & aggregate ---
df = pd.read_csv(path_to_tables + f"/{CSV}.csv")
print(df['scenario'].unique())

PERIODS = [2030, 2040, 2050]

SCENARIOS = df['scenario'].unique()

# SCENARIOS = ["VRE & ESS (high) Coal (high)",
#              "VRE & ESS (high) Coal (low)",
#              "VRE & ESS (low) Coal (high)",
#              "VRE & ESS (low) Coal (low)",
#              "VRE & ESS (mid) Coal (high)",
#              "VRE & ESS (mid) Coal (low) - REF"]

TECH_ORDER = ["Coal", 
              "Gas", 
              "Diesel", 
              "Nuclear", 
              "Other", 
              "Hydro", 
              "Solar", 
              "Wind", 
              "Pumped Storage",
              "Battery", 
              "Hydrogen"]

def _fmt_int(x: float) -> str:
    # integers without thousands separators to match your example
    return f'{round(float(x)):,}'

def _latex_escape(s: str) -> str:
    # minimal escaping for table cells
    return (str(s)
            .replace("\\", r"\textbackslash{}")
            .replace("&", r"\&")
            .replace("%", r"\%")
            .replace("_", r"\_")
            .replace("#", r"\#"))

def _vals_for(scenario: str, technology: str, metric: str):
    sub = agg[(agg["scenario"] == scenario) & (agg["technology"] == technology)]
    by_p = {p: 0 for p in PERIODS}
    for _, r in sub.iterrows():
        p = int(r["period"])
        if p in by_p:
            by_p[p] = r[metric]
    return [_fmt_int(by_p[p]) for p in PERIODS]

def _totals_for(scenario: str, metric: str):
    sub = agg[(agg["scenario"] == scenario) & (agg["period"].isin(PERIODS))]
    sums = sub.groupby("period")[metric].sum()
    return [_fmt_int(sums.get(p, 0)) for p in PERIODS]


# Sum across all zones and statuses
agg = (df.groupby(["scenario", 
                   "technology", 
                   "period"], as_index = False)[["capacity_mw", 
                                                 "capacity_mwh"]].sum())

# --- Build LaTeX ---
lines = []
lines.append(r"\begin{tabular}{l|ccc|ccc}")
lines.append(r"\toprule")
lines.append(r"\multicolumn{1}{c|}{\multirow{2}{*}{\textbf{Scenario \& Technology}}} & \multicolumn{3}{c|}{\textbf{Capacity (MW)}}  &  \multicolumn{3}{c}{\textbf{Capacity (MWh)}} \\")
lines.append(r"\multicolumn{1}{c|}{} & \textbf{2030} & \textbf{2040} & \textbf{2050} & \textbf{2030} & \textbf{2040} & \textbf{2050} \\")

for scen in SCENARIOS:
    lines.append(r"\midrule")
    mw_tot   = _totals_for(scen, "capacity_mw")
    mwh_tot  = _totals_for(scen, "capacity_mwh")
    scen_tex = _latex_escape(scen)
    lines.append(
        rf"\textbf{{{scen_tex}}} & "
        rf"\textbf{{{mw_tot[0]}}} & \textbf{{{mw_tot[1]}}} & \textbf{{{mw_tot[2]}}} & "
        rf"\textbf{{{mwh_tot[0]}}} & \textbf{{{mwh_tot[1]}}} & \textbf{{{mwh_tot[2]}}} \\"
    )
    lines.append(r"\midrule ")
    for tech in TECH_ORDER:
        mw       = _vals_for(scen, tech, "capacity_mw")
        mwh      = _vals_for(scen, tech, "capacity_mwh")
        tech_tex = _latex_escape(tech)
        lines.append(
            rf"{tech_tex:14} & {mw[0]} & {mw[1]} & {mw[2]}  & {mwh[0]} & {mwh[1]} & {mwh[2]} \\"
        )

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

latex_table = "\n".join(lines)

# # Write to file and also print (so you can copy-paste)
Path(path_to_tables + f"/{CSV}.tex").write_text(latex_table, encoding="utf-8")
#print(latex_table)

['100% Carbon Target' 'No Carbon Target' '80% Carbon Target'
 '90% Carbon Target ' 'No Clean Target or Carbon Target']


4590